In [ ]:
import time,shapely
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from shapely.geometry import Polygon
from scipy.ndimage import binary_dilation as dilation
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
import rasterio as rio

In [ ]:
level_dimensions = snakemake.params.level_dimensions
desired_regions = snakemake.params.desired_regions

In [ ]:
overlap_paths = {
    'iucn_filter' : snakemake.input.cdda_overlap,
    'natura_filter' : snakemake.input.natura_overlap,
    'birds_bats_filter' : snakemake.input.bird_overlap,
    'forests_filter' : snakemake.input.forest_overlap,
}

In [ ]:
def env_constraint_defs(lvl):
    env={}
    for level in snakemake.wildcards.level: # Iterate over low, medium and high.
        lvl_dim = level_dimensions.get(level)
        e = {}
        for item in lvl_dim: # iterate over the relevant datasets in each level
            if lvl_dim[item] == 'overlap': # 
                path=overlap_paths.get(item)
                lvl_dim[item] = path
            e[item] = lvl_dim[item]
        env[level] = e
    return env[lvl]

In [ ]:
def get_europe():
    europe = (
        gpd
        .read_file(snakemake.input.euroshape)
        .query("CNTR_CODE == @desired_regions")
        .dissolve(by='CNTR_CODE')
        .loc[:,['geometry']]
    ) 
    return europe.to_crs("EPSG:3035")

In [ ]:
def get_country_codes():
    country_codes = {
    "Austria" : "AT",
    "Belgium" : "BE",
    "Bulgaria" : "BG", #
    "Switzerland" : "CH", #
    "Czech Republic" : "CZ", #
    "Germany" : "DE",
    "Denmark" : "DK",
    "Estonia" : "EE" ,
    "Spain" : "ES",
    "Finland" : "FI", #
    "France" : "FR",
    "United Kingdom" : "UK",
    "Greece" : "GR",
    "Croatia" : "HR", #
    "Hungary" : "HU", #
    "Ireland" : "IE",
    "Italy" : "IT",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Latvia" : "LV",
    "Netherlands" : "NL",
    "Norway" : "NO",
    "Poland" : "PL", #
    "Portugal" : "PT",
    "Romania" : "RO", #
    "Sweden" : "SE",
    "Slovenia" : "SI", #
    "Slovakia" : "SK" #
    }
    return country_codes

In [ ]:
def _as_transform(x, y):
    lx, rx = x[[0, -1]]
    ly, uy = y[[0, -1]]

    dx = float(rx - lx) / float(len(x) - 1)
    dy = float(uy - ly) / float(len(y) - 1)

    return rio.Affine(dx, 0, lx - dx / 2, 0, dy, ly - dy / 2)

def padded_transform_and_shape(bounds, res):
    """
    Get the (transform, shape) tuple of a raster with resolution `res` and
    bounds `bounds`.
    """
    left, bottom = ((b // res) * res for b in bounds[:2])
    right, top = ((b // res + 1) * res for b in bounds[2:])
    shape = int((top - bottom) / res), int((right - left) / res)
    return rio.Affine(res, 0, left, 0, -res, top), shape

def get_bounding():
    rectx1 = -12
    rectx2 = 44
    recty1 = 33
    recty2 = 72
    
    polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
    )
    polygon=shapely.segmentize(polygon, max_segment_length=0.5)
    b=gpd.GeoDataFrame(geometry=[polygon],crs="EPSG:4326")
    return b.to_crs("EPSG:3035")

In [ ]:
def build_cdda(iucn):
    d=(gpd.read_file(snakemake.input.cdda)
       .query("iucnCategory in @iucn")
       .loc[:,["geomType","geometry"]])
    # buffer points by 100m
    d.loc[d["geomType"]=="point","geometry"]=d.loc[d["geomType"]=="point",:].buffer(100)
    return d

In [ ]:
def build_cdda_by_country(by_country,thresh=15.0,min_wf=5):
    country_codes=get_country_codes()
    by_country=(pd.read_csv(by_country,skiprows=1)
                .loc[:,["country",
                        "Ia (% pixels)",	
                         "Ib (% pixels)",
                         "II (% pixels)",	
                         "III (% pixels)",	
                         "IV (% pixels)",	
                         "V (% pixels)",	
                         "VI (% pixels)",	
                         "notReported (% pixels)",	
                         "notAssigned (% pixels)",
                         "notApplicable (% pixels)",
                         "total_windfarms"]]
                )
    by_country.country=by_country.country.replace(country_codes)
    shp=(gpd.read_file(snakemake.input.cdda)
         .replace({"EL": "GR"}))
    cats=shp.iucnCategory.unique()
    
    cdda_out=[]
    for _,row in by_country.iterrows():
        cnt=row.country
        for cat in cats: 
            if row[f"{cat} (% pixels)"] < thresh and row["total_windfarms"] > min_wf:
                cdda_out.append(
                    shp.query("cddaRegionCode == @cnt and iucnCategory == @cat").loc[:,["cddaRegionCode","iucnCategory","geomType","geometry"]])
            
            # if a country only has a small number of wfs then assume all cats are excluded
            elif row["total_windfarms"] < min_wf:
                cdda_out.append(
                    shp.query("cddaRegionCode == @cnt and iucnCategory == @cat").loc[:,["cddaRegionCode","iucnCategory","geomType","geometry"]])
            else:
                print(f"No exclusion for: {cat} in {cnt}")
        
    cdda_out=pd.concat(cdda_out)
    cdda_out.loc[cdda_out["geomType"]=="point","geometry"]=cdda_out.loc[cdda_out["geomType"]=="point",:].buffer(100)
    return cdda_out

In [ ]:
def build_filter(by_country,cols,thresh=15.0,min_wf=5):
    country_codes=get_country_codes()
    by_country=(pd.read_csv(by_country,skiprows=1)
                .loc[:,cols])
    by_country.country=by_country.country.replace(country_codes)
    sel=(by_country.iloc[:,1] > min_wf) & (by_country.iloc[:,2] < thresh)
    return by_country.country[sel]

def build_filter_bb(by_country,thresh=15.0,min_wf=10):
    country_codes=get_country_codes()
    by_country=pd.read_csv(by_country,skiprows=1)
    by_country.country=by_country.country.replace(country_codes)
    by_country=by_country.set_index("country",drop=True)
    sel_wf=(by_country["Num_windFarms"] > min_wf)
    sel=(by_country.loc[sel_wf,"60":"90"] < thresh)
    return sel.loc[sel.any(axis=1),:]

In [ ]:
def build_natura(natura_by_country,thresh=15.0,min_wf=5):
    
    shp=gpd.read_file(snakemake.input.natura2000)
    country_codes=get_country_codes()
    by_country=(pd.read_csv(natura_by_country,skiprows=1)
                .loc[:,["country","ABC (% pixels)","total_windfarms"]])
    by_country.country=by_country.country.replace(country_codes)
    nat_out=[]
    for _,(cnt,pct,tot_area) in by_country.iterrows():
        if pct < thresh and tot_area > min_wf:
            nat_out.append(shp.query("MS == @cnt").loc[:,["MS","geometry"]])
        elif tot_area < min_wf:
            nat_out.append(shp.query("MS == @cnt").loc[:,["MS","geometry"]])
        else:
            print(f"No exclusion for: natura2k in {cnt}")
    return pd.concat(nat_out)


In [ ]:
def env_build_by_country():
    lvls=snakemake.wildcards
    thresh=15.
    for lvl in lvls:
        c=env_constraint_defs(lvl)
        dst_transform, shape=padded_transform_and_shape(rio.features.bounds(get_bounding()), 100)
        out=np.zeros(shape, np.uint8) 
        to_raster=[]
        if type(c["iucn_filter"])==list:
            print("Exclude CDDA based on list")
            to_raster.append(build_cdda(c["iucn_filter"]))
        else:
            print("Exclude country specific CDDA")
            to_raster.append(build_cdda_by_country(c["iucn_filter"],thresh=thresh))
        if c["natura"]:
            if c["natura_filter"]=="all":
                print("Exclude all natura2k")
                shp=gpd.read_file(snakemake.input.natura2000)
                to_raster.append(shp)
            else:
                print("Exclude country specific natura2k")
                to_raster.append(build_natura(c["natura_filter"],thresh=thresh))
                
        for shp in to_raster:
            shp["area"]=1
            shp["area"]=shp["area"].astype("uint8")
        
            out=out+rio.features.rasterize(zip(shp.geometry,shp["area"].values),
                               out_shape=shape,
                               transform=dst_transform,
                               fill=0,
                               all_touched=True)
            
        # if requested add buffer for cdda and natura
            
        if c["buffer"] !=0:
            iterations = int(c["buffer"] / 100) + 1
            out = dilation(out, iterations=iterations).astype("uint8")
        
        if c["birds_bats"]:
            print("Adding birds and bats exclusion")
            if c["birds_bats_filter"]=="all":
                print("All countries at same quantile")
                europe=get_europe()
                src=rio.open(snakemake.input.birds)
                src_data = src.read(1)
                src_transform = src.transform
                temp=np.zeros(src_data.shape, dtype=np.uint8)
                for cnt,geom in europe.iterrows():
                    in_cnt=~geometry_mask(geom,
                                   src_data.shape,
                                   src_transform,
                                   all_touched=False)
                    
                    cnt_data=src_data.copy()
                    cnt_data[~in_cnt]=np.nan
                    quant=np.nanpercentile(cnt_data,c["birds_bats_quantile"])
                    temp[in_cnt & (src_data > quant)]=1
                destination = np.zeros(shape, np.uint8)
                rio.warp.reproject(temp,
                                     destination,
                                     src_transform=src_transform,
                                     src_crs=src.crs,
                                     dst_transform=dst_transform,
                                     dst_crs=src.crs,
                                     resampling=Resampling.nearest,
                                     num_threads=2)
        
                out=out+destination
            else:    
                print("Excluding country specific BB")
                cnt_sel=build_filter_bb(c["birds_bats_filter"],thresh=thresh)
                europe=get_europe().query("CNTR_CODE in @cnt_sel.index")
                src=rio.open(snakemake.input.birds)
                src_data = src.read(1)
                src_transform = src.transform
                temp=np.zeros(src_data.shape, dtype=np.uint8)
                for cnt,geom in europe.iterrows():
                    quantile=cnt_sel.columns[cnt_sel.loc[cnt_sel.index==cnt,:].values[0]][0]
                    in_cnt=~geometry_mask(geom,
                                   src_data.shape,
                                   src_transform,
                                   all_touched=False)
                    
                    cnt_data=src_data.copy()
                    cnt_data[~in_cnt]=np.nan
                    quant=np.nanpercentile(cnt_data,float(quantile))
                    temp[in_cnt & (src_data > quant)]=1
                destination = np.zeros(shape, np.uint8)
                rio.warp.reproject(temp,
                                     destination,
                                     src_transform=src_transform,
                                     src_crs=src.crs,
                                     dst_transform=dst_transform,
                                     dst_crs=src.crs,
                                     resampling=Resampling.nearest,
                                     num_threads=2)
                out=out+destination
        if c["peat"]:
            print("Excluding peat")
            src=rio.open(snakemake.input.corine)
            src_data=src.read(1)
            src_transform = src.transform
            sel=(src_data==36)
            src_data=np.where(sel,1,0)
            peat = np.zeros(shape, np.uint8)
            rio.warp.reproject(src_data,
                             peat,
                             src_transform=src_transform,
                             src_crs=src.crs,
                             dst_transform=dst_transform,
                             dst_crs=src.crs,
                             resampling=Resampling.nearest,
                             num_threads=2)
            out=out+peat
            
        if c["forests"]:
            if c["forests_filter"]=="all":
                print("Excluding all forests")
                src=rio.open(snakemake.input.corine)
                src_data=src.read(1)
                src_transform = src.transform
                # removed agro forests
                sel=((src_data==22) | (src_data==23) | (src_data==24) | (src_data==25))
                src_data=np.where(sel,1,0)
                forests = np.zeros(shape, np.uint8)
                rio.warp.reproject(src_data,
                                 forests,
                                 src_transform=src_transform,
                                 src_crs=src.crs,
                                 dst_transform=dst_transform,
                                 dst_crs=src.crs,
                                 resampling=Resampling.nearest,
                                 num_threads=2)
                out=out+forests
                
            else:
                print("Excluding country specific Forests")
                cnt_sel=build_filter(c["forests_filter"],["country","Num of WF","Total"],thresh=thresh)
                europe=get_europe().query("CNTR_CODE in @cnt_sel")
                src=rio.open(snakemake.input.corine)
                src_data = src.read(1)
                src_transform = src.transform
                temp=np.zeros(src_data.shape, dtype=np.uint8)
                sel=((src_data==22) | (src_data==23) | (src_data==24) | (src_data==25))
                for cnt,geom in europe.iterrows():
                    in_cnt=~geometry_mask(geom,
                                   src_data.shape,
                                   src_transform,
                                   all_touched=False)
                    
                    temp[in_cnt & sel]=1
                forests = np.zeros(shape, np.uint8)
                rio.warp.reproject(temp,
                                     forests,
                                     src_transform=src_transform,
                                     src_crs=src.crs,
                                     dst_transform=dst_transform,
                                     dst_crs=src.crs,
                                     resampling=Resampling.nearest,
                                     num_threads=2)
                out=out+forests
        out[out>=1]=1
            
        with rio.open(
            snakemake.output.envExcl,
            mode="w",
            driver="GTiff",
            height=out.shape[0],
            width=out.shape[1],
            count=1,
            dtype=out.dtype,
            #dtype=np.uint8,
            #crs=pc.rio.crs,
            crs="EPSG:3035",
            transform=dst_transform,
            compress='lzw',
            ) as new_dataset:
                new_dataset.write(out, 1)

In [ ]:
env_build_by_country()